# Fitting QENS data with a Lorentzian

Quasi-elastic neutron scattering (QENS) measures small energy exchanges between neutrons and a sample. A
broadening of the elastic line, described by a Lorentzian, reveals how fast the atoms or molecules move. This
notebook fits a Lorentzian, convolved with the resolution of the instrument, to reduced QENS data, and then
works out quantities that follow from the fit, such as the relaxation time, with their uncertainty, using the
arithmetic and the functions of `es.math`.

Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and
open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import pooch
from scipy.stats import norm

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Fitter
from easyscience import Parameter
from easyscience.base_classes import ModelBase

The function `show` prints a labelled number as value, uncertainty and unit.

In [ ]:
def show(label, number):
    """Print a labelled number as value ± uncertainty with its unit."""
    print(f'{label} = {number.value:.4g} ± {number.error or 0.0:.2g} {number.unit}')

## The data

The data are simulated and reduced intensities $I(\omega)$ with their uncertainties, as a function of the
energy transfer $\omega$ in meV. They are stored in a plain text file with three columns, which is
downloaded once and then read from the local copy. Only the region around the elastic line, $|\omega| <
0.06$ meV, is kept:

In [ ]:
filename = pooch.retrieve(
    url='https://public.esss.dk/groups/scipp/dmsc-summer-school/2025/4-reduction/energy_transfer_QENS_unknown_quasi_elastic_many_neutrons.dat',
    known_hash='sha256:e49fa9a1d2ef5eeb714524903e8ffa9de6616e5a299a799cbb0b2f3ee8fd459a',
)
omega, intensity, error = np.loadtxt(filename, unpack=True)
keep = np.isfinite(intensity) & (omega > -0.06) & (omega < 0.06)
omega, intensity, error = omega[keep], intensity[keep], error[keep]

print(f'{omega.size} points, omega from {omega.min():.3f} to {omega.max():.3f} meV')

The chart is interactive: hover over a point to read its values. It loads Plotly from the internet, so it
needs a connection. The charts are shown with a small helper, `show_figure(fig)`, which is defined in a
hidden cell; in your own notebook `fig.show()` does the same.

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=omega,
        y=intensity,
        mode='markers',
        name='data',
        error_y=dict(type='data', array=error, visible=True),
        marker=dict(size=4),
    )
)
fig.update_layout(
    xaxis_title='energy transfer / meV',
    yaxis_title='I(ω)',
    margin=dict(l=60, r=20, t=30, b=50),
)
show_figure(fig)

## The model

QENS data are usually described by a Lorentzian,

$$
L(\omega) = \frac{A\,\gamma}{\pi\left[(\omega - \omega_0)^2 + \gamma^2\right]},
$$

where $A$ is the area, $\gamma$ the half width at half maximum and $\omega_0$ a small offset of the centre.
The instrument blurs the line, which is modelled by convolving it with a Gaussian of width $\sigma$. The
model has four parameters. The widths and the offset are energies, so they carry the unit meV.

Inside `__call__` the model works with plain NumPy arrays and the `.value` of each parameter: `es.math` takes
one `Parameter` at a time, so it belongs **after** the fit, on the fitted parameters.

In [ ]:
class Quasielastic(ModelBase):
    """A Lorentzian convolved with a Gaussian resolution, as a function of the energy transfer."""

    def __init__(self, area: float, gamma: float, omega_0: float, sigma: float):
        super().__init__()
        self._area = Parameter(display_name='area', value=area, min=0.01, max=100)
        self._gamma = Parameter(display_name='gamma', value=gamma, unit='meV', min=1e-4, max=1e-2)
        self._omega_0 = Parameter(display_name='omega_0', value=omega_0, unit='meV', min=0, max=2e-3)
        self._sigma = Parameter(display_name='sigma', value=sigma, unit='meV', min=1e-5, max=1e-1)

    @property
    def area(self) -> Parameter:
        return self._area

    @property
    def gamma(self) -> Parameter:
        return self._gamma

    @property
    def omega_0(self) -> Parameter:
        return self._omega_0

    @property
    def sigma(self) -> Parameter:
        return self._sigma

    def __call__(self, omega):
        width = self._gamma.value
        lorentzian = (
            self._area.value / np.pi * width / ((omega - self._omega_0.value) ** 2 + width**2)
        )
        resolution = norm(0, self._sigma.value).pdf(omega)
        resolution /= resolution.sum()
        return np.convolve(lorentzian, resolution, 'same')

## The fit

The fit starts from a rough guess. The weights are the inverse uncertainties of the points. After the fit each parameter
holds its fitted value and its uncertainty:

In [ ]:
model = Quasielastic(area=10.0, gamma=8e-3, omega_0=1e-3, sigma=1e-3)
fitter = Fitter(model, model)
result = fitter.fit(x=omega, y=intensity, weights=1 / error)

print(f'converged: {result.success}, reduced chi-squared: {result.reduced_chi2:.2f}')
show('area', model.area)
show('gamma', model.gamma)
show('omega_0', model.omega_0)
show('sigma', model.sigma)

The reduced chi-squared is close to 1, so the model describes the data within their uncertainties. The width
$\gamma$ is determined well, to about 2 %. The resolution width $\sigma$ is not: its uncertainty is about half
of its value. This matters below.

In [ ]:
fit_omega = np.linspace(omega.min(), omega.max(), 400)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=omega,
        y=intensity,
        mode='markers',
        name='data',
        error_y=dict(type='data', array=error, visible=True),
        marker=dict(size=4),
    )
)
fig.add_trace(
    go.Scatter(x=fit_omega, y=model(fit_omega), mode='lines', name='fit', line=dict(color='black'))
)
fig.update_layout(
    xaxis_title='energy transfer / meV',
    yaxis_title='I(ω)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.75, y=0.95),
)
show_figure(fig)

## From a width to a time

The width $\gamma$ of the Lorentzian is the inverse of a relaxation time, $\tau = \hbar/\gamma$. The reduced
Planck constant in meV·ps is a constant without uncertainty, so it is a `DescriptorNumber`. The units of
`meV*ps` and `meV` cancel to give a time, and the uncertainty of $\gamma$ is carried along:

In [ ]:
hbar = DescriptorNumber(display_name='hbar', value=0.6582119569, unit='meV*ps', variance=0.0)

tau = hbar / model.gamma
show('relaxation time', tau)

The intermediate scattering function, which describes how correlations decay in time, is $I(t) =
\exp(-t/\tau)$. With `es.math.exp` its value at a chosen time comes with an uncertainty. The exponent has to
be dimensionless: a time in ps divided by $\tau$ in ps is:

In [ ]:
time = DescriptorNumber(display_name='time', value=10.0, unit='ps', variance=0.0)

decay = es.math.exp(-(time / tau))
show('I(10 ps)', decay)

Computed for many times, the same expression gives the decay curve with its uncertainty band:

In [ ]:
times = np.linspace(0.0, 300.0, 61)

values, errors = [], []
for value in times:
    point = DescriptorNumber(display_name='time', value=value, unit='ps', variance=0.0)
    decay = es.math.exp(-(point / tau))
    values.append(decay.value)
    errors.append(decay.error or 0.0)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=times,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 uncertainty',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(x=times, y=values, mode='lines', name='I(t)', line=dict(color='rgb(99, 110, 250)'))
)
fig.update_layout(
    xaxis_title='time / ps',
    yaxis_title='I(t)',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.65, y=0.95),
)
show_figure(fig)

### Exercise 1: The half-life of the decay

The intermediate scattering function falls to one half at $t_{1/2} = \tau \ln 2$. Work out $t_{1/2}$ for the fitted width, in ps, with its uncertainty.

In [ ]:
hbar = DescriptorNumber(display_name='hbar', value=0.6582119569, unit='meV*ps', variance=0.0)
two = DescriptorNumber(display_name='two', value=2.0, variance=0.0)

<details class="question">
<summary>Hint 1</summary>

The relaxation time is `hbar / model.gamma`.

</details>

<details class="question">
<summary>Hint 2</summary>

The constant 2 is wrapped in `two`, because `es.math.log` takes a `Parameter` or a `DescriptorNumber`.

</details>

In [ ]:
%%solution The half-life is $\ln 2 \approx 0.69$ times the relaxation time.
tau = hbar / model.gamma
t_half = tau * es.math.log(two)
show('half-life', t_half)

### Exercise 2: The decay after 50 ps

Work out the intermediate scattering function $I(t)$ at $t = 50$ ps for the fitted width, with its uncertainty.

In [ ]:
hbar = DescriptorNumber(display_name='hbar', value=0.6582119569, unit='meV*ps', variance=0.0)
time = DescriptorNumber(display_name='time', value=50.0, unit='ps', variance=0.0)

<details class="question">
<summary>Hint 1</summary>

The argument of the exponential is $-t/\tau$ with $\tau = \hbar/\gamma$.

</details>

<details class="question">
<summary>Hint 2</summary>

The ratio of a time in ps to another time in ps has no unit, which is what `es.math.exp` needs.

</details>

In [ ]:
%%solution The uncertainty is small because $\gamma$ is known to about 2 %.
tau = hbar / model.gamma
decay = es.math.exp(-(time / tau))
show('I(50 ps)', decay)

## The line width and the resolution

The full width at half maximum of the Lorentzian is $\Gamma = 2\gamma$, and its peak height, before the
blurring by the instrument, is $A/(\pi\gamma)$. The Gaussian resolution has the full width at half maximum
$2\sqrt{2\ln 2}\;\sigma$. With the constant wrapped in a `DescriptorNumber`, `es.math.sqrt` and `es.math.log`
give it with its uncertainty, which `convert_unit` expresses in µeV:

In [ ]:
two = DescriptorNumber(display_name='two', value=2.0, variance=0.0)

resolution = 2 * es.math.sqrt(2 * es.math.log(two)) * model.sigma
resolution.convert_unit('ueV')
show('resolution (FWHM)', resolution)

line_width = 2 * model.gamma
line_width.convert_unit('ueV')
show('line width (FWHM)', line_width)

### Exercise 3: The height of the line

The peak height of the Lorentzian, before the blurring by the resolution, is $A/(\pi\gamma)$. Work it out with its uncertainty.

In [ ]:
area, gamma = model.area, model.gamma

<details class="question">
<summary>Hint 1</summary>

A plain number such as `np.pi` can multiply a `Parameter`. The height has the unit of $1/\text{meV}$, because the area has none.

</details>

<details class="question">
<summary>Hint 2</summary>

The uncertainty of the area and that of the width both enter the uncertainty of the height.

</details>

In [ ]:
%%solution The height is the area divided by $\pi\gamma$, so its relative uncertainty combines those of both.
height = area / (np.pi * gamma)
show('peak height', height)

### Exercise 4: Resolution against line width

Work out the ratio of the resolution FWHM to the line width $\Gamma = 2\gamma$, with its uncertainty. Is the line broader than the resolution?

In [ ]:
two = DescriptorNumber(display_name='two', value=2.0, variance=0.0)

<details class="question">
<summary>Hint 1</summary>

The two widths are the resolution full width at half maximum, $2\sqrt{2\ln 2}\;\sigma$, and the line width $2\gamma$. Compute both as in the text above.

</details>

<details class="question">
<summary>Hint 2</summary>

Both are energies, so their ratio is a dimensionless number.

</details>

In [ ]:
%%solution The ratio is well below 1, so the line is broader than the resolution. Its uncertainty is large because $\sigma$ is poorly determined.
resolution = 2 * es.math.sqrt(2 * es.math.log(two)) * model.sigma
line_width = 2 * model.gamma
ratio = resolution / line_width
show('resolution / line width', ratio)

## Good to know

* **Constants have to be wrapped.** `es.math` accepts a `Parameter` or a `DescriptorNumber`, not a plain
  number, and it takes one value at a time, not an array. That is why the model uses NumPy and the results
  use `es.math`:

In [ ]:
for argument in (2, np.array([1.0, 2.0])):
    try:
        es.math.log(argument)
    except Exception as exc:
        print(f'{type(exc).__name__}: {exc}')

* **A quantity that is poorly determined stays poorly determined.** The relative uncertainty of $\sigma$
  decides the relative uncertainty of everything that is derived from it:

In [ ]:
print(f'relative uncertainty of sigma: {model.sigma.error / model.sigma.value:.0%}')
print(f'relative uncertainty of the resolution (FWHM): {resolution.error / resolution.value:.0%}')
print(f'relative uncertainty of gamma: {model.gamma.error / model.gamma.value:.1%}')

* **The uncertainty is symmetric and first order.** For a quantity that is a strongly non-linear function of
  a poorly determined parameter, such as $\hbar/\sigma$, a symmetric uncertainty can be misleading. The Bayesian notebook compares it with the full posterior. It also shows that $\gamma$ and $\sigma$ are correlated, which the arithmetic of EasyScience ignores.